# 02 LangChain Core

Messages, prompts, models, and structured output, the foundation beneath every agent.


In [1]:
import sys
import importlib
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "shared").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

for _name in (
    "shared.dataflow",
    "shared.notebook_display",
    "shared.llm",
    "shared.bootcamp_fixtures",
):
    importlib.reload(importlib.import_module(_name))

print(f"Project root: {ROOT}")


c:\Users\Azooo\langchain-bootcamp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: c:\Users\Azooo\langchain-bootcamp


## Learning objectives

- Invoke models with `ChatPromptTemplate`
- Use `with_structured_output()` with `json_mode` on DeepSeek
- Understand message types and LCEL piping


In [2]:
import os

def env_status():
    keys = {
        "DEEPSEEK_API_KEY": bool(os.getenv("DEEPSEEK_API_KEY")),
        "DEEPSEEK_MODEL": os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash"),
        "LLM_PROVIDER": os.getenv("LLM_PROVIDER", "deepseek"),
        "TAVILY_API_KEY": bool(os.getenv("TAVILY_API_KEY")),
        "LANGSMITH_API_KEY": bool(os.getenv("LANGSMITH_API_KEY")),
    }
    for k, v in keys.items():
        print(f"{k}: {v}")
    return keys

ENV = env_status()


ZAI_API_KEY: True
OPENAI_API_KEY: False
ANTHROPIC_API_KEY: False
TAVILY_API_KEY: False
LANGSMITH_API_KEY: True
LLM_PROVIDER: deepseek


In [3]:
import os
from shared.llm import get_model

def require_llm():
    return get_model()


## Rendered prompt messages

Before calling the model, inspect what the prompt template produces.


Run the demo below to verify the behavior.


In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise travel assistant."),
    ("user", "Suggest one {region} trip under ${budget}."),
])
rendered = prompt.invoke({"region": "Europe", "budget": 900})
for i, msg in enumerate(rendered.messages):
    print(f"[{i}] {type(msg).__name__}: {msg.content}")


[0] SystemMessage: You are a concise travel assistant.
[1] HumanMessage: Suggest one Europe trip under $900.


**Reflect:** Placeholders `{region}` and `{budget}` are replaced before the model sees the text.


## Model response

Pipe prompt to model with LCEL (`prompt | model`).


Run the demo below to verify the behavior.


In [5]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise travel assistant."),
    ("user", "Suggest one {region} trip under ${budget}."),
])
model = require_llm()
chain = prompt | model
response = chain.invoke({"region": "Europe", "budget": 900})
print("Type:", type(response).__name__)
print("Content:", response.content[:400])


Type: AIMessage
Content: **Lisbon, Portugal** – 5 days, ~$850  
- **Flights**: ~$350 (round-trip from major US cities, off-peak)  
- **Hostel**: ~$150 (5 nights)  
- **Food**: ~$150 (local meals, pastéis de nata)  
- **Activities**: ~$100 (tram 28, Belém Tower, free miradouros)  
- **Transport**: ~$50 (metro, day trips to Sintra)  
- **Misc**: ~$50 (souvenirs, tips)  

Budget-friendly, vibrant culture, and affordable eats


## Structured output schema

Pydantic models define the shape the model must return.


Run the demo below to verify the behavior.


In [6]:
from pydantic import BaseModel, Field
import json

class DestinationSuggestion(BaseModel):
    name: str = Field(description="City or region")
    country: str
    budget_needed: int
    why: str

print("Schema the model sees:")
print(json.dumps(DestinationSuggestion.model_json_schema(), indent=2)[:500], "...")


Schema the model sees:
{
  "properties": {
    "name": {
      "description": "City or region",
      "title": "Name",
      "type": "string"
    },
    "country": {
      "title": "Country",
      "type": "string"
    },
    "budget_needed": {
      "title": "Budget Needed",
      "type": "integer"
    },
    "why": {
      "title": "Why",
      "type": "string"
    }
  },
  "required": [
    "name",
    "country",
    "budget_needed",
    "why"
  ],
  "title": "DestinationSuggestion",
  "type": "object"
} ...


## Structured output result

Wrap the model with `with_structured_output()` to get typed objects.


Run the demo below to verify the behavior.


In [8]:
from pydantic import BaseModel, Field

class DestinationSuggestion(BaseModel):
    name: str = Field(description="City or region")
    country: str
    budget_needed: int
    why: str


model = require_llm()
structured = model.with_structured_output(DestinationSuggestion, method="json_mode")
out = structured.invoke("Budget beach destination under $800")
print(out.model_dump_json(indent=2))


BadRequestError: Error code: 400 - {'error': {'message': 'This response_format type is unavailable now', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}